# 1 · det_objects — 지금 앞에 무엇이 있는가

전방 ±60° 안의 도로 사용자를 클래스와 위치로 열거합니다. 한 순간을 묻기 때문에 미래 지평선 제약이 없고, 그래서 앵커를 3/6/9/12/15/18초 여섯 곳에 둘 수 있습니다.

이 노트북은 네 가지를 확인합니다 — **어떤 원시 데이터에서**, **어떤 코드를 거쳐**, **무엇이 입력으로 들어가고**, **빌드된 파일이 그 코드와 일치하는지**. GPU 는 필요 없습니다.

In [ ]:
import os, sys, json, textwrap
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import numpy as np
import pandas as pd

from datatools import paths

ITEMS = os.path.join(paths.COMMON_DIR, "instruct_items_tasks01_06.parquet")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 80)
wrap = lambda s, i="   ": textwrap.fill(str(s), 94, initial_indent=i,
                                        subsequent_indent=i)
VARIANTS = ['det_objects_azdeg', 'det_objects_3dbbox']
print("variants:", VARIANTS)

## 1. 어떤 원시 데이터에서 오는가

| 아카이브 | 주기 | 읽는 것 |
|---|---|---|
| `obstacle.offline` | 10 Hz | center_x/y/z, size_*, orientation_*, label_class, track_id, reference_frame='rig' |
| `egomotion` | 10 Hz | x, y, z, qx..qw — rig 좌표를 월드로 올리는 데 필요 |
| `radar (lrr1/mrr2/srr0)` | 20 또는 12.7 Hz | azimuth, elevation, distance, radial_velocity, rcs, doppler_ambiguity |
| `camera intrinsics` | 클립당 1 | f-theta 다항식 — CoT 의 카메라 방위각용 |

## 2. 어떤 코드를 거치는가

실행 순서입니다.

| 함수 | 하는 일 |
|---|---|
| `boxes_world(obstacle, ego)` | rig 좌표에서 거리·방위각, 월드 좌표에서 이동 여부 |
| `visible_at(boxes, t_s)` | ±0.15 s 창 → ±60°/300 m 섹터 → 트랙당 1관측 → 거리순 |
| `head(MAX_LISTED=8)` | 가까운 순 8개 |
| `describe_object / describe_object_xyz` | 문장화 |
| `object_evidence(scan, listed)` | CoT 용: 박스 안 반사점의 수·거리·방위각 |
| `camera_azimuth(box centre)` | CoT 용: 이미지 박스에서 역산한 방위각 |

전부 `datatools/frame_objects.py` 와 `datatools/geometry.py` 에 있습니다.

In [ ]:
import inspect
from datatools import frame_objects as F
for name in ['boxes_world', 'visible_at', 'head', 'describe_object / describe_object_xyz', 'object_evidence', 'camera_azimuth']:
    fn = getattr(F, name, None)
    if fn is None:
        from datatools import geometry as G
        fn = getattr(G, name, None)
    if fn is None or not callable(fn):
        print(f'{name}: (모듈 함수 아님)'); continue
    doc = (inspect.getdoc(fn) or '').split(chr(10))[0]
    print(f'{name:34s} {doc[:80]}')

## 3. 입력으로 무엇이 들어가는가

**비전 1장 (질문한 순간) · 레이더 20스캔 (직전 약 1초) · ego 1샘플**

입력 창은 태스크마다 다릅니다. 로더의 `WINDOWS` 표가 그것을 정하고, 레이더는 항상 20스캔이라 창의 길이가 바뀌면 샘플링 속도가 따라 바뀝니다 — 인코더 입력 모양은 변하지 않습니다.

In [ ]:
from training.instruct_data import WINDOWS, INSTANT_TASKS, WINDOW_TASKS
for v in VARIANTS:
    for name in (v, v + "_cot"):
        if name in WINDOWS:
            secs, hz, frames = WINDOWS[name]
            print(f"{name:26s} 창 {secs}초 · 레이더 {hz} Hz × 20스캔 · 비전 {frames}장")
        elif name in INSTANT_TASKS:
            print(f"{name:26s} 순간 — 비전 1장 · 레이더 20스캔/1초 · ego 1")
        else:
            print(f"{name:26s} 클립 전체 — 비전 20장 · 레이더 20스캔/20초")

## 4. 실제 아이템

빌드된 파일에서 그대로 꺼냅니다.

In [ ]:
built = pd.read_parquet(ITEMS)
for v in VARIANTS:
    sub = built[built.task == v]
    if sub.empty:
        print(f"{v}: 파일에 없음"); continue
    r = sub.iloc[0]
    print("=" * 96)
    print(f"{v}   clip {r.clip_id[:8]}  frame {r.frame} (t={r.frame-1}s)  split {r.split}")
    print("Q:"); print(wrap(r.prompt))
    print("A:"); print(wrap(r.target))

## 5. CoT — 근거가 답을 만드는가

`_cot` 변형은 `{"rationale": ..., "answer": ...}` 입니다. **근거를 따라가면 답이 나와야** 합니다. 나오지 않으면 그 사슬은 잘못된 것이고, 보상을 걸면 모델이 그 잘못된 사슬을 배웁니다.

In [ ]:
for v in VARIANTS:
    name = v + "_cot"
    sub = built[built.task == name] if 'built' in dir() else None
    if sub is None or sub.empty:
        continue
    r = sub.iloc[0]
    d = json.loads(r.target)
    print("=" * 96); print(name)
    print("R:"); print(wrap(d["rationale"]))
    print("A:"); print(wrap(d["answer"]))

## 6. 보상

평가 채점기에서 유도했습니다. 정답을 그대로 넣으면 1.0 이 나와야 하고, 내용을 망가뜨리면 떨어져야 합니다.

In [ ]:
import re
from training.task_scorers import reward_for

def wreck(text):
    """형식은 두고 숫자만 2배로."""
    return re.sub(r"\d+(?:\.\d+)?",
                  lambda m: str(round(float(m.group()) * 2, 1)), text)

rows = []
for v in VARIANTS:
    for name in (v, v + "_cot"):
        fn = reward_for(name)
        if fn is None:
            continue
        sub = built[built.task == name] if 'built' in dir() else None
        if sub is None or sub.empty:
            continue
        t = sub.iloc[0].target
        rows.append({"task": name, "reward": fn.__name__,
                     "정답": round(fn(t, t), 3),
                     "숫자 2배": round(fn(wreck(t), t), 3)})
pd.DataFrame(rows)

## 7. 데이터 양

`val` 은 `train` 에 합쳐져 있습니다 — 클립 분할이 train 86,607 / val 54,163 / test 37,121 인데, 모델 선택은 `test` 에서 하므로 검증용 3분의 1이 쓰이지 않고 있었습니다.

In [ ]:
from collections import Counter
from training.instruct_data import load_items
names = [v for v in VARIANTS] + [v + "_cot" for v in VARIANTS]
rows = []
for split in ("train", "test"):
    c = Counter(i["task"] for i in load_items(tuple(names), split))
    for n in names:
        rows.append({"task": n, "split": split, "items": c.get(n, 0)})
pd.DataFrame(rows).pivot(index="task", columns="split", values="items")

## 8. 이 태스크에서 내린 결정과 근거

**이동 표기를 뺐다**

라벨의 moved 는 트랙 전체 20초 변위인데 입력은 1프레임 + 1초입니다. 레이더가 보는 물체만 놓고도 그 순간의 도플러와 85.8% 만 일치했고, 나머지는 입력으로 도달할 수 없는 정답이었습니다. 이동 판정은 06 이 맡습니다.

**상위 8개 절단**

앞에 물체가 12개 있어도 8개만 적히므로, 9번째를 맞혀도 오탐이 됩니다.

**오토라벨**

source='scene:obstacles:autolabels:v2'. 사람이 검증한 텍스트는 QA 뿐입니다.